Feature 1: Data Continuity Checker
Problem: In field deployments, data gaps occur due to connectivity issues, device reboots, or
meter communication failures. We need to detect and report these gaps.
Task:
1. Write a Python script that analyzes timestamps in the data log
2. Expected reading interval: ~5 minutes. Flag gaps > 10 minutes as potential issues
3. For each device (Shed_01, Shed_02), report: total readings, missing intervals, data
completeness %
4. Identify if gaps are device-specific or system-wide (affecting both sheds)
5. Output: Summary report with gap timestamps and duration
Evaluation Criteria:
• Correct time-series handling
• Accurate gap detection
• Per-device analysis
• Practical insights from the analysis

In [51]:
import pandas as pd
path = "vireon_sample_data.tsv"
df = pd.read_csv(path,sep="\t")
df.head()

,Timestamp,Date,Time,Hour,Device_ID,Location,Meter_Serial,Model_Number,ToD_Period,Rate_Rs_kWh,...,Daily_Energy_Wh,PF_Rebate_Pct,Effective_Rate,Daily_Cost_Rs,Carbon_kg_CO2,Pi_Temp_C,Pi_CPU_Pct,Pi_Mem_Pct,Pi_Disk_Pct,Pi_Uptime_Hrs
0,2026-02-12 22:14:48,2026-02-12,22:14:48,22,vireon-cortex-cl01-mu-01,Shed_01,24057940303,4405,PEAK,8.37,...,477920,8,7.70,3054.29,41134.2,37.3,0.0,6.6,24.6,9.3
1,2026-02-12 22:18:11,2026-02-12,22:18:11,22,vireon-cortex-cl01-mu-02,Shed_02,24057940XXX,4405,PEAK,8.37,...,44209,0,8.37,254.37,3349.9,47.2,0.0,11.0,25.4,1.1
2,2026-02-12 22:20:20,2026-02-12,22:20:20,22,vireon-cortex-cl01-mu-01,Shed_01,24057940303,4405,PEAK,8.37,...,478044,7,7.78,3055.25,41134.3,38.4,0.0,6.7,24.6,9.4
3,2026-02-12 22:23:41,2026-02-12,22:23:41,22,vireon-cortex-cl01-mu-02,Shed_02,24057940XXX,4405,PEAK,8.37,...,44209,0,8.37,254.37,3349.9,47.7,0.0,11.0,25.4,1.2
4,2026-02-12 22:25:49,2026-02-12,22:25:49,22,vireon-cortex-cl01-mu-01,Shed_01,24057940303,4405,PEAK,8.37,...,478172,8,7.70,3056.24,41134.4,38.4,0.0,6.7,24.6,9.5


In [21]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['timestamp', 'date', 'time', 'hour', 'device_id', 'location',
       'meter_serial', 'model_number', 'tod_period', 'rate_rs_kwh',
       'watts_total', 'watts_r', 'watts_y', 'watts_b', 'kw_total', 'va_total',
       'va_r', 'va_y', 'va_b', 'kva_total', 'pf_avg', 'pf_r', 'pf_y', 'pf_b',
       'vll_avg', 'v_ry', 'v_yb', 'v_br', 'vln_avg', 'v_r', 'v_y', 'v_b',
       'current_total', 'current_r', 'current_y', 'current_b',
       'neutral_current_a', 'fire_risk_level', 'frequency_hz', 'energy_wh',
       'energy_kwh', 'energy_vah', 'energy_vah_derived',
       'voltage_unbalance_pct', 'current_unbalance_pct', 'load_pct',
       'run_hours', 'max_demand_kva', 'max_demand_kw', 'session_energy_wh',
       'daily_energy_wh', 'pf_rebate_pct', 'effective_rate', 'daily_cost_rs',
       'carbon_kg_co2', 'pi_temp_c', 'pi_cpu_pct', 'pi_mem_pct', 'pi_disk_pct',
       'pi_uptime_hrs'],
      dtype='str')

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["device_id"].unique()

<StringArray>
['vireon-cortex-cl01-mu-01', 'vireon-cortex-cl01-mu-02']
Length: 2, dtype: str

In [47]:
for device in df["device_id"].unique():
    d = df[df["device_id"] == device]
    for shed in df["location"].unique():
        s = d[d["location"] == shed]
        print(s["timestamp"])

0    2026-02-12 22:14:48
2    2026-02-12 22:20:20
4    2026-02-12 22:25:49
6    2026-02-12 22:31:20
8    2026-02-12 22:36:50
10   2026-02-12 22:42:21
12   2026-02-12 22:47:51
14   2026-02-12 22:53:21
16   2026-02-12 22:58:52
18   2026-02-12 23:04:22
20   2026-02-12 23:09:53
22   2026-02-12 23:15:23
24   2026-02-12 23:20:53
26   2026-02-12 23:26:24
28   2026-02-12 23:31:54
30   2026-02-12 23:37:24
32   2026-02-12 23:42:55
34   2026-02-12 23:48:25
36   2026-02-12 23:53:56
38   2026-02-12 23:59:26
40   2026-02-13 00:04:57
42   2026-02-13 00:10:26
44   2026-02-13 00:15:56
46   2026-02-13 00:21:27
48   2026-02-13 00:26:57
50   2026-02-13 00:32:27
52   2026-02-13 00:37:58
54   2026-02-13 00:43:28
56   2026-02-13 00:48:58
58   2026-02-13 00:54:28
60   2026-02-13 00:59:58
Name: timestamp, dtype: datetime64[us]
Series([], Name: timestamp, dtype: datetime64[us])
Series([], Name: timestamp, dtype: datetime64[us])
1    2026-02-12 22:18:11
3    2026-02-12 22:23:41
5    2026-02-12 22:29:12
7    2026

In [50]:
for shed in df["location"].unique():
    d = df[df["location"] == shed]
    gap = d["timestamp"].diff().dt.total_seconds() / 60
    danger = gap[gap > 10]
    print(danger)
    completness = 100 - (len(danger)/len(d))*100
    print("Shed:", shed)
    print("Ttal reading: ", len(d))
    print("Missing: ",len(danger))
    print("%", completness)

Series([], Name: timestamp, dtype: float64)
Shed: Shed_01
Ttal reading:  31
Missing:  0
% 100.0
Series([], Name: timestamp, dtype: float64)
Shed: Shed_02
Ttal reading:  31
Missing:  0
% 100.0
